### RAG Pipeline - Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path 
import sys; sys.path.append("..")
from core import act_name

/var/folders/zf/r06rj4550hq2y6gchrcvw8mw0000gn/T/ipykernel_45479/575303202.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
                doc.metadata['act'] = act_name(pdf_file.name)
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data/pdf")

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/AKIOKI+Times-Roman', '/FontDescriptor': IndirectObject(11, 0, 4600293296), '/Type': '/Font', '/FirstChar': 32, '/LastChar': 151, '/Widths': [250, 0, 0, 0, 0, 0, 0, 0, 333, 333, 500, 0, 250, 333, 250, 0, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 278, 278, 0, 0, 0, 0, 0, 722, 667, 667, 722, 611, 556, 722, 722, 333, 389, 722, 611, 889, 722, 722, 556, 0, 667, 556, 611, 722, 722, 944, 722, 722, 0, 333, 0, 333, 0, 500, 0, 444, 500, 444, 500, 444, 333, 500, 500, 278, 278, 500, 278, 778, 500, 500, 500, 500, 333, 389, 278, 500, 500, 722, 500, 500, 444, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 333, 333, 444, 444, 0, 0, 1000], '/Encoding': '/WinAnsiEncoding', '/Subtype': '/Type1'}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Ba

Found 11 PDF files to process

Processing: 202504081796627129.pdf


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/AOEUYK+Times-Italic', '/FontDescriptor': IndirectObject(25, 0, 4600293296), '/Type': '/Font', '/FirstChar': 32, '/LastChar': 120, '/Widths': [250, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 250, 0, 250, 0, 0, 500, 500, 500, 500, 500, 500, 500, 500, 500, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 722, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 500, 500, 444, 500, 444, 278, 500, 500, 278, 278, 444, 278, 722, 500, 500, 500, 0, 389, 389, 278, 0, 444, 0, 444], '/Encoding': '/WinAnsiEncoding', '/Subtype': '/Type1'}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/TFCOBE+Times-Bold', '/FontDescriptor': IndirectObject(27, 0, 4600293296), '/Type': '/Font', '/FirstChar': 32, '/LastChar': 151, '/Widths': [250, 0, 0, 0, 0, 0, 0, 0, 0,

  ✓ Loaded 19 pages

Processing: A187209.pdf
  ✓ Loaded 53 pages

Processing: Shop-and-Establishment-Act-Handbook.pdf
  ✓ Loaded 9 pages

Processing: b0620548445580767b5c0d18c95c26f7.pdf
  ✓ Loaded 116 pages

Processing: transferofproperty.pdf
  ✓ Loaded 69 pages

Processing: Bharatiya_Nyaya_Sanhita,_2023.pdf
  ✓ Loaded 137 pages

Processing: the-sale-of-goods-act-1930.pdf
  ✓ Loaded 21 pages

Processing: negotiable-instrument-act.pdf
  ✓ Loaded 32 pages

Processing: 0e6aea16d11ed1a622f1de43ff3710b4.pdf
  ✓ Loaded 28 pages

Processing: 682a44b5426bff2c1f4943ee1b2fd566.pdf
  ✓ Loaded 56 pages

Processing: A190805.pdf
  ✓ Loaded 347 pages

Total documents loaded: 887


In [3]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [4]:
chunks = split_documents(all_pdf_documents)
for d in chunks:
    d.page_content = f"[{d.metadata['act']}]\n{d.page_content}"
print(chunks[100].page_content[:200])   # should start with the Act name

Split 887 documents into 3495 chunks

Example chunk:
Content: 1 
 
THE LEGAL SERVICES AUTHORITIES ACT, 1987 
________ 
ARRANGEMENT OF SECTIONS 
________ 
CHAPTER I 
PRELIMINARY 
SECTIONS 
1.  Short title, extent and commencement.  
2.  Definitions.  
 
CHAPTER I...
Metadata: {'producer': 'GPL Ghostscript 9.06', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2019-11-13T16:53:56+05:30', 'moddate': '2019-11-13T16:53:56+05:30', 'title': 'Microsoft Word - The Legal Service Authorities Act, 1987.docx', 'author': 'Lenovo', 'source': '../data/pdf/202504081796627129.pdf', 'total_pages': 19, 'page': 0, 'page_label': '1', 'source_file': '202504081796627129.pdf', 'file_type': 'pdf', 'act': 'Legal Services Authorities Act, 1987'}
[Indian Contract Act, 1872]
9 
 
SECTIONS 
234. Consequence of inducing agent or principal to act on belief that principal or agent will be held 
exclusively liable. 
235. Liability of pretended agent


## Embedding and vector store DB

In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

/Users/nainishkothari/Documents/Kuku/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1107 15:50:25.726000 45479 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [6]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6584.46it/s]


Model loaded successfully. Embedding dimension: 384


In [7]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "legal_docs", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Indian legal PDFs for RAG", "hnsw:space": "cosine"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: legal_docs
Existing documents in collection: 3495


In [8]:
if vectorstore.collection.count() > 0:
    vectorstore.collection.delete(ids=vectorstore.collection.get()["ids"])
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 3495 texts...


Batches: 100%|██████████| 110/110 [00:30<00:00,  3.55it/s]


Generated embeddings with shape: (3495, 384)
Adding 3495 documents to vector store...
Successfully added 3495 documents to vector store
Total documents in collection: 3495


In [9]:
import sys; sys.path.append("..")
import os; os.environ["VECTOR_DIR"] = "../data/vector_store"
import core
law = core.Lawyer(os.environ["GROQ_API_KEY"])
for q in ["employer not paying salary for two months",
          "refund not given for returned product",
          "landlord not returning security deposit"]:
    print("\n", q)
    for h in law.retrieve(q, 5, 0.0):
        print(h["score"], h["act"], "p", h["page"], "|", h["text"][:90].replace("\n"," "))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6271.61it/s]



 employer not paying salary for two months
0.46 Code on Social Security, 2020 p 54 | [Code on Social Security, 2020] months immediately preceding the accident, been in the ser
0.45 Code on Social Security, 2020 p 54 | [Code on Social Security, 2020] shall, in addition to the amount of the arrears and intere
0.42 Code on Social Security, 2020 p 54 | [Code on Social Security, 2020] (c) in other cases including cases in which it is not poss
0.38 TODO: name of this Act (56 pages) p 29 | [TODO: name of this Act (56 pages)] which the money became due to the worker from the empl
0.37 TODO: name of this Act (56 pages) p 33 | [TODO: name of this Act (56 pages)] time during normal working hours at least once a day; 

 refund not given for returned product
0.37 A190805.pdf p 126 | [A190805.pdf] preferred, when the appeal has been disposed of:  1[Provided that a document
0.36 Indian Contract Act, 1872 p 40 | [Indian Contract Act, 1872] bailee is to receive no remuneration, the bailor shall repay 